# Classificação de sentimentos com corpus de brinquedo

Vamos modelizar um conjunto fictício de resenhas de clientes sobre determinado restaurante. Além dos textos, temos à disposição uma etiqueta para cada resenha informando se a opinião do cliente soa positiva ou negativa.

A partir das probabilidades calculadas, será possível gerar uma função que avalie novas resenhas, classificando-as como positivas ou negativas.

In [2]:
# Estas são as resenhas para a modelização:
corpus = [
    ('Esse restaurante é um lixo', 'NEG'),
    ('A comida servida no inferno', 'NEG'),
    ('Lugar porco, mais que porco, sórdido', 'NEG'),
    ('Perfeito para o seu cachorro', 'POS'),
    ('Restaurante ótimo', 'POS')
    ]

# E esta é a resenha com a qual vamos testar o classificador mais tarde.
# Você pode testar com outras, também.
teste = '''
        Restaurante horroroso.
        A comida é lixo em estado coloidal!
        O outro emprego do garçom é carcereiro de masmorra.
        '''


**Pré-processamento:**

*   Tokenização
*   Limpeza
* Filtragem de stop words

In [3]:
# Módulos usados mais adiante

import itertools
from collections import Counter
from numpy import prod
from math import log


In [4]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download('stopwords')
stops = nltk.corpus.stopwords.words('portuguese')

# OBS: Nas classificações, o "não" é importante. Vale tirá-lo da lista de stops
stops.remove('não')


# Tokenização de sentenças
def tokenizar_sentencas(txt: str) -> list:
    txt = txt.replace('\n', ' ')
    return sent_tokenize(txt, language='portuguese')


# Tokenização de palavras e símbolos
def tokenizar(txt: str) -> list:
    return word_tokenize(txt, language='portuguese')


def limpar(lista: list) -> list:
    return [i.lower() for i in lista if i.isalpha()]


def sem_stops(lst_palavras: list) -> list:
    return [p for p in lst_palavras if p not in stops]


def achatar(lista: list) -> list:
    return list(itertools.chain(*lista))


def pre_processar(str_texto: str) -> list:
    return sem_stops(limpar(tokenizar(str_texto)))


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\knd\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\knd\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [5]:
corpus = [(pre_processar(i[0]), i[1]) for i in corpus]
corpus


[(['restaurante', 'lixo'], 'NEG'),
 (['comida', 'servida', 'inferno'], 'NEG'),
 (['lugar', 'porco', 'porco', 'sórdido'], 'NEG'),
 (['perfeito', 'cachorro'], 'POS'),
 (['restaurante', 'ótimo'], 'POS')]

# Sua vez!
**Separação dos documentos**

*   Extração de duas listas separadas contendo documentos etiquetados como "negativos" e "positivos"
*   Extração do vocabulário total do corpus (juntando as palavras das resenhas positivas e das negativas). Atenção! Lembre-se de que o vocabulário são as palavras sem repetição.


In [7]:
# Sua solução:
pos = [i for i in corpus if i[1] == 'POS']
neg = [i for i in corpus if i[1] == 'NEG']

pos, neg


([(['perfeito', 'cachorro'], 'POS'), (['restaurante', 'ótimo'], 'POS')],
 [(['restaurante', 'lixo'], 'NEG'),
  (['comida', 'servida', 'inferno'], 'NEG'),
  (['lugar', 'porco', 'porco', 'sórdido'], 'NEG')])

In [ ]:

vocab = set(achatar([i[0] for i in corpus]))
vocab


{'cachorro',
 'comida',
 'inferno',
 'lixo',
 'lugar',
 'perfeito',
 'porco',
 'restaurante',
 'servida',
 'sórdido',
 'ótimo'}

In [9]:
achatar(pos)


[['perfeito', 'cachorro'], 'POS', ['restaurante', 'ótimo'], 'POS']

# Sua vez!
**Contagens**

*   *N* documentos negativos
*   *N* documentos positivos
* *N* total de documentos

* *N* itens no vocabulário
* Contagens de ocorrências de cada palavra nas resenhas negativas. **Dica:** use a função `Counter()` do módulo `collection`s para isso.
* Contagens de ocorrências de cada palavra nas resenhas positivas.
* *N* total de palavras "negativas"
* *N* total de palavras "positivas"



In [10]:
# Sua solução:

# Contagens de documentos (resenhas)
n_docs_pos = len(pos)
n_docs_neg = len(neg)
n_docs = len(corpus)


# N itens no vocabulário
n_vocab = len(vocab)


# Contagens de cada palavra (dicionários de ocorrências)
cont_pos = Counter(achatar([i[0] for i in pos]))
cont_neg = Counter(achatar([i[0] for i in neg]))

# N total de palavras em cada classe
n_total_pos = len(set(achatar([i[0] for i in pos])))
n_total_neg = len(set(achatar([i[0] for i in neg])))

# Print
print('Contagem de documentos positivos:', n_docs_pos)
print('Contagem de documentos negativos:', n_docs_neg)
print('Contagem total de documentos:', n_docs)

print('Tamanho do vocabulário:', n_vocab)

print('Contagem de cada palavra positiva:', cont_pos)
print('Contagem de cada palavra negativa:', cont_neg)

print('Contagem total de palavras positivas:', n_total_pos)
print('Contagem total de palavras negativas:', n_total_neg)


Contagem de documentos positivos: 2
Contagem de documentos negativos: 3
Contagem total de documentos: 5
Tamanho do vocabulário: 11
Contagem de cada palavra positiva: Counter({'perfeito': 1, 'cachorro': 1, 'restaurante': 1, 'ótimo': 1})
Contagem de cada palavra negativa: Counter({'porco': 2, 'restaurante': 1, 'lixo': 1, 'comida': 1, 'servida': 1, 'inferno': 1, 'lugar': 1, 'sórdido': 1})
Contagem total de palavras positivas: 4
Contagem total de palavras negativas: 8


# Sua vez!
**Cálculo das probabilidades de classificação**

Há duas probabilidades a calcular, uma para cada classe ($c$): a probabilidade de uma resenha qualquer ser negativa e a probabilidade de ela ser falsa.

Essa probabilidade é dada por:

\begin{equation}
P(c) \prod\limits_{i=1}^n P(f_{i} | c)
\end{equation}

Onde a probabilidade de cada atributo $f$ é dada pelo número de ocorrências (tokens) desse atributo dividido pelo número de tokens em cada classe.

**Dica:** para calcular o produtório de uma lista, use a função `prod` do módulo `numpy`.


Vamos suavizar a classificação com o método de Laplace, isto é, somando 1 a cada atributo no numerador e somando também a cardinalidade do vocabulário ao denominador.

A probabilidade isolada (anterior) da classe $c$ é dada por:

\begin{equation}
	P(c) = \dfrac{contagem(c)}{N}
\end{equation}

Já a probabilidade suavizada de um atributo qualquer pertencer a $c$ é:

\begin{equation}
	P(f_{i} | c) = \dfrac{contagem(f_{i},c) + 1}{contagem(c) + V}
 \end{equation}

Por fim, para evitar o underflow aritmético, é sempre uma boa ideia calcular probabilidades encadeadas com logaritmos. A diferença com documentos e vocabulários pequenos não é grande, mas tende a ser quando se trabalha com dados reais.

\begin{equation}
\hat{c} = \underset{c \in \mathcal{C}}{\operatorname{argmax}} \ \log P(c) + \sum\limits_{i=1}^n \log P(f_{i} | c)
\end{equation}

Agora, com base nessas informações, procure classificar a mensagem-teste apresentada acima.

In [ ]:
# Sua solução

pi_pos = n_docs_pos / n_docs
pi_neg = n_docs_neg / n_docs

psi_pos = n_total_pos / (n_total_pos + n_total_neg)
psi_neg = n_total_neg / (n_total_pos + n_total_neg)

# Probabilidades de cada palavra em cada classe
def prob_palavra(palavra: str, classe: str) -> float:
    if classe == 'POS':
        return (cont_pos[palavra] + 1) / (n_total_pos + n_vocab)
    elif classe == 'NEG':
        return (cont_neg[palavra] + 1) / (n_total_neg + n_vocab)
    

---

# **Tarefa em sala:** Detecção de spam com corpus de dados reais

Nessa tarefa, você vai trabalhar com parte do corpus de mensagens de e-mail da Enron.

As mensagens estão em arquivos de texto curtos em inglês (principalmente, mas não só) e têm anotações manuais no título e na primeira linha de texto com sua etiqueta. As classes são "spam" (etiqueta 1) ou "ham" (0).

Aqui está uma mensagem de exemplo:


---


0

Subject: meter 1517 - jan 1999
george ,
i need the following done :
jan 13
zero out 012 - 27049 - 02 - 001 receipt package id 2666
allocate flow of 149 to 012 - 64610 - 02 - 055 deliv package id 392
jan 26
zero out 012 - 27049 - 02 - 001 receipt package id 3011
zero out 012 - 64610 - 02 - 055 deliv package id 392
these were buybacks that were incorrectly nominated to transport contracts
( ect 201 receipt )
let me know when this is done
hc

---

Observe o 0 na primeira linha. Ele indica que a mensagem foi etiquetada como *ham*.

Você deve:



1.   Abrir cada arquivo de texto.
2.   Pré-processar os dados, implementando os seguintes procedimentos:

*   Tokenização
* Eliminação de stop words
*   Limpeza e homogeneização dos tokens
* Stemização

Use o NLTK para gerar a lista de stop words em inglês e para implementar um stemizador também em inglês:

---

```
import nltk
stops = nltk.corpus.stopwords.words('english')
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer('english')

def stemizar(lista):
    return [stemmer.stem(i) for i in lista]
```

---

3. Dividir o corpus pré-processado em treinamento (80%) e teste (20%).

4. Usando o corpus de treinamento, realizar as contagens do vocabulário, das classes (*spam* ou *ham*) e dos atributos (cada token corresponde a um atributo).

5. Calcular as probabilidades relacionadas às contagens.

6. Implementar uma função bayes() que receba uma mensagem e devolva a probabilidade de ela ser classificada como *spam* e como *ham*:

`return prob_spam, prob_ham`

7. Nessa função, não deixe de incluir:

* Uma condicional para testar se as palavras a classificar fazem parte do vocabulário de treinamento. As que não fazem devem ser simplesmente ignoradas (ficar de fora do cálculo).

* A suavização de Laplace.

8. Classificar todo o corpus de teste passando cada mensagem pela função bayes().

9. Avaliar a performance do classificador: para cada mensagem, comparar a classificação com as etiquetas de *spam* ou *ham* e gerar uma lista com os resultados dessa avaliação em termos de Verdadeiro Positivo (VP), Verdadeiro Negativo (VN), Falso Positivo (FP) e Falso Negativo (FN).

10. Com base nessa lista, calcular:

* precisao = vp / (vp + fp)
* cobertura = vp / (vp + fn)
* acuracia = (vp + vn) / (vp + vn + fp + fn)
* Medida_F = 2 * (precisao * cobertura) / (precisao + cobertura)








**Pré-processamento**

Para facilitar as coisas, aqui vai o código para abrir o arquivo compactado, ler as mensagens individualmente e gerar a lista de treinamento e a de teste.

In [ ]:
from google.colab import files
import glob

# Aqui, você deve escolher o arquivo ZIP com as mensagens em seu disco rígido
arquivo = files.upload()
!unzip 'Enron.zip' -d 'enron'


In [ ]:
from collections import Counter
import itertools
import nltk
nltk.download('stopwords')
stops = nltk.corpus.stopwords.words('english')
stops.remove('no')
stops.remove('not')

nltk.download('punkt')
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer('english')


def tokenizar(str_texto: str) -> list:
    return word_tokenize(str_texto)

def limpar(lista: list) -> list:
    return [i.lower() for i in lista if i.isalpha()]

def sem_stops(lista: list) -> list:
    return [i for i in lista if i not in stops]

def stemizar(lista: list) -> list:
    return [stemmer.stem(i) for i in lista]

def achatar(lista: list) -> list:
    return list(itertools.chain(*lista))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
arqs = glob.glob('enron/*.txt')
mensagens = list()
for arq in arqs:
    arquivo = open(arq, 'r')
    classe = arquivo.readline()[0]  # Pega só o número e deixa de fora o \n
    if classe == '1':
        classe = 'SPAM'
    else:
        classe = 'HAM'

    texto = arquivo.read()
    texto = stemizar(sem_stops(limpar(tokenizar(texto))))
    mensagens.append((texto, classe))
    arquivo.close()

treinamento = mensagens[:round(len(mensagens) * 0.8)]
teste = mensagens[len(treinamento) + 1:]

spam = [t[0] for t in treinamento if t[1] == 'SPAM']
ham = [t[0] for t in treinamento if t[1] == 'HAM']


Pronto! O resto é com você. Divirta-se!